# Building a Custom MCP Server with FastMCP and OpenAI Agents SDK

Most of the examples out there show how to connect an agent to a *pre-built* MCP server. But what if you need to build the server yourself from scratch?

The idea is simple: we'll use [FastMCP](https://gofastmcp.com) to write the server side. It handles all the boilerplate — you just write a Python function, and the schema, validation, and protocol logic are generated automatically. Then, we'll use the [OpenAI Agents SDK](https://github.com/openai/openai-agents-python) to attach it to an agent over standard I/O.

Here's what we cover:
- Building a task-management MCP server with `create`, `list`, and `complete` tools
- Connecting it to an agent using `MCPServerStdio`
- Extending it with the handoff pattern (a triage agent routing to our MCP specialist)

In [ ]:
%pip install openai-agents fastmcp

_What just happened?_ We installed the two main libraries we need. `fastmcp` is what we use to build the server, and `openai-agents` is the SDK that handles the actual AI agents talking to it.

### 0. Setup

Make sure your API key is locked in before proceeding. (Side note: we apply `nest_asyncio` here so we don't trip over Jupyter's existing event loop when running our `await` calls later).

In [ ]:
import os
import sys
import nest_asyncio
nest_asyncio.apply()

if not os.environ.get("OPENAI_API_KEY"):
    print("Set OPENAI_API_KEY to run this notebook")


_What just happened?_ We just made sure your environment has the keys it needs to talk to OpenAI. (Side note: `nest_asyncio` just stops Jupyter from throwing an event loop error when we run asynchronous `await` code in the cells later).

### 1. Building the Server

First up, the custom server itself. FastMCP makes this painless — you just write standard Python functions and slap an `@mcp.tool` decorator on them. The docstrings and type hints automatically turn into the JSON schema that the LLM reads.

*(Side note: We're using `%%writefile` to dump this into `task_server.py`. The stdio transport needs a standalone script file to launch as a subprocess, so keeping it inline in Jupyter doesn't work out of the box without saving it to disk first.)*

In [ ]:
%%writefile task_server.py
from fastmcp import FastMCP
from datetime import datetime

mcp = FastMCP("Task Manager")

# In-memory task storage
tasks: dict[str, dict] = {}
counter: int = 0


@mcp.tool
def create_task(title: str, description: str = "") -> str:
    """Create a new task with the given title and optional description."""
    global counter
    counter += 1
    task_id = f"T{counter:04d}"
    tasks[task_id] = {
        "id": task_id,
        "title": title,
        "description": description,
        "status": "pending",
        "created_at": datetime.now().isoformat(),
    }
    return f"Created task {task_id}: {title}"


@mcp.tool
def list_tasks(status: str = "") -> str:
    """List all tasks, optionally filtered by status (pending or completed)."""
    if not tasks:
        return "No tasks found."
    lines = []
    for t in tasks.values():
        if status and t["status"] != status:
            continue
        lines.append(f"[{t['id']}] {t['title']} - {t['status']}")
    return "\n".join(lines) if lines else "No matching tasks."


@mcp.tool
def complete_task(task_id: str) -> str:
    """Mark a task as completed by its task ID."""
    if task_id not in tasks:
        return f"Error: Task {task_id} not found."
    tasks[task_id]["status"] = "completed"
    return f"Task {task_id} ({tasks[task_id]['title']}) marked as completed."


if __name__ == "__main__":
    mcp.run()

_What just happened?_ We wrote a completely functional MCP server in about 30 lines of code. We defined a dictionary `tasks` to hold our data in memory, and created three functions (`create_task`, `list_tasks`, `complete_task`). By dropping `@mcp.tool` on top of them, FastMCP automatically translates those functions into the exact JSON format that LLMs expect to see. It's essentially magic.

### 2. Wiring it to an Agent

Now that `task_server.py` is sitting on disk, we can wire it up. 

We'll use `MCPServerStdio` to fire up the server as a subprocess and talk to it over standard I/O. Once the server is running, we just pass it into the `mcp_servers` list when instantiating our `Agent`. (We also drop a `trace()` in here so you can check the exact payloads on the platform dashboard).

In [ ]:
from agents import Agent, Runner, gen_trace_id, trace
from agents.mcp import MCPServerStdio


async def main():
    async with MCPServerStdio(
        name="Task Manager MCP",
        params={"command": sys.executable, "args": ["task_server.py"]},
    ) as server:
        agent = Agent(
            name="Task Assistant",
            model="gpt-4o",
            instructions="You are a helpful task management assistant. Use the MCP tools to create, list, and complete tasks for the user.",
            mcp_servers=[server],
        )

        trace_id = gen_trace_id()
        with trace(workflow_name="MCP Task Manager", trace_id=trace_id):
            print(f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}\n")

            # Create a task and list all tasks
            result = await Runner.run(
                starting_agent=agent,
                input="Create a task for 'Review Q3 budget report' and then show me all tasks.",
            )
            print("=== Result ===")
            print(result.final_output)


await main()

_What just happened?_ We used `MCPServerStdio` to run that `task_server.py` script in the background. Then we created an `Agent` and told it, "Hey, you have access to this server." When we asked it to create a task, the agent automatically understood what tools the server had, fired off the right JSON to create the task, and read the response back. No manual API routing required on your end.

### 3. Multi-Agent Handoffs

A single agent with tools is fine, but real-world setups usually need specialists. So let's implement the handoff pattern. 

The idea here is to build a user-facing `Triage Agent`. If the user wants to mess with tasks, it routes them to our MCP-backed Task Manager. If they just have a general question, the triage agent handles it directly. No messy nested tool calls — the specialist just takes over the conversation cleanly.

In [ ]:
from agents import Agent, Runner
from agents.mcp import MCPServerStdio


async def handoff_demo():
    async with MCPServerStdio(
        name="Task Manager MCP",
        params={"command": sys.executable, "args": ["task_server.py"]},
    ) as server:
        # Specialist agent backed by the MCP server
        task_agent = Agent(
            name="Task Manager",
            model="gpt-4o",
            instructions="You manage tasks using the available MCP tools. Handle all task-related requests the user makes.",
            mcp_servers=[server],
        )

        # Triage agent that routes to the specialist
        triage_agent = Agent(
            name="Triage Agent",
            model="gpt-4o",
            instructions=(
                "You are a triage agent. Determine what the user needs:\n"
                "- For task management requests (create, list, or complete tasks), hand off to the Task Manager agent.\n"
                "- For anything else, answer directly."
            ),
            handoffs=[task_agent],
        )

        # Test 1: task request -> routed to specialist
        result = await Runner.run(
            starting_agent=triage_agent,
            input="I need to create a task for 'Update team documentation'.",
        )
        print("=== Triage — Task Request ===")
        print(result.final_output)
        print()

        # Test 2: general question -> handled by triage directly
        result = await Runner.run(
            starting_agent=triage_agent,
            input="What is the capital of France?",
        )
        print("=== Triage — General Question ===")
        print(result.final_output)


await handoff_demo()

_What just happened?_ We created two separate agents. The `Triage Agent` is the front desk — it looks at what the user wants. When we asked it to create a task, it saw the `handoffs=[]` list and immediately handed the microphone over to the `Task Manager` agent (which actually holds the connection to our MCP server). When we asked a random trivia question, the triage agent just answered it directly without bothering the specialist.

## Summary

That's the loop. We started from scratch and ended up with a multi-agent system talking to a custom MCP server. 

To recap:
- **The server:** We built `task_server.py` using FastMCP, letting decorators handle the schema generation.
- **The connection:** We used `MCPServerStdio` to spin it up and handed it to a specialist agent.
- **The routing:** We set up a triage agent that intelligently hands off task requests to the specialist.

If you want to take this further, you could try hooking the server up to a persistent SQLite db, adding a `delete_task` tool, or swapping from stdio to Streamable HTTP so the server can run remotely.